<a href="https://colab.research.google.com/github/fidlarsyn/grokking-Deep-Learning-Chapter-11-to-15/blob/main/15_Deep_Learning_on_Unseen_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **The Problem of Privacy in Deep Learning**

*Deep learning* bertumpu pada ketersediaan data latih untuk mempelajari pola dan korelasi. Namun, dalam banyak kasus penggunaan nyata yang paling berharga (seperti aplikasi kesehatan, keuangan, dan data pribadi), data tersebut bersifat sangat sensitif. Pendekatan tradisional mengharuskan pengumpulan dan pemusatan seluruh data pengguna ke satu server pusat (*centralized server*), yang memicu risiko kebobolan privasi dan hambatan regulasi hukum.

# **Federated Learning**

Diperkenalkan oleh Google pada tahun 2017, **Federated Learning** membalikkan paradigma pemusatan data: alih-alih mengirimkan dataset pengguna ke server pusat, **model dikirimkan ke lokasi data berada** (perangkat lokal pengguna). Pelatihan dilakukan secara lokal pada perangkat masing-masing pengguna, dan hanya perbaikan/pembaruan bobot model (*weight updates*) yang dikirimkan kembali ke server. Pengguna mempertahankan kendali penuh atas data mereka tanpa perlu membagikan data mentah ke pihak mana pun.

# **Learning to Detect Spam**

Sebagai studi kasus praktis, bab ini menggunakan dataset email Enron (*spam* dan *ham*) untuk melatih model klasifikasi *spam*. Data di-praproses dengan mengonversi kata-kata teks menjadi barisan indeks numerik serta menyelaraskan panjang email menjadi 500 kata melalui penambahan token $<unk>$ (padding).

# **Let’s Make It Federated**

Simulasi *Federated Learning* dilakukan dengan membagi dataset email secara terpisah ke beberapa pengguna (misalnya Bob, Alice, dan Sue). Pada setiap ronde pelatihan, server mengirimkan salinan model awal ke masing-masing pengguna. Setiap pengguna melatih model pada data lokal mereka, lalu server mengumpulkan model baru dari Bob, Alice, dan Sue untuk dihitung rata-rata bobotnya (`model.weight.data = (bob_model + alice_model + sue_model) / 3`).

In [ ]:
import copy

# Pembagian data lokal pengguna
bob = (train_data[0:1000], train_target[0:1000])
alice = (train_data[1000:2000], train_target[1000:2000])
sue = (train_data[2000:], train_target[2000:])

# Federated Training Loop
for i in range(3):
    print("Starting Training Round...")
    print("\\tStep 1: send the model to Bob")
    bob_model = train(copy.deepcopy(model), bob, bob[22], iterations=1)

    print("\\n\\tStep 2: send the model to Alice")
    alice_model = train(copy.deepcopy(model), alice, alice[22], iterations=1)

    print("\\n\\tStep 3: Send the model to Sue")
    sue_model = train(copy.deepcopy(model), sue, sue[22], iterations=1)

    print("\\n\\tAverage Everyone's New Models")
    model.weight.data = (bob_model.weight.data + \\
                         alice_model.weight.data + \\
                         sue_model.weight.data) / 3

    print("\\t% Correct on Test Set: " + str(test(model, test_data, test_target) * 100))

# **Hacking into Federated Learning**

Meskipun data mentah tidak dikirim ke server, *Federated Learning* dasar masih memiliki celah keamanan. Jika seorang pengguna melatih model menggunakan *batch* kecil atau sampel individual, pemilik server atau peretas dapat memeriksa selisih bobot sebelum dan sesudah pelatihan (`bobs_model.weight.data - model.weight.data`). Dari nilai perubahan bobot yang tidak bernilai nol, peretas dapat merekonstruksi kata-kata rahasia yang ada dalam email pengguna (seperti kata kunci/password).

In [ ]:
import copy
import numpy as np

# Bob melatih email rahasia
bobs_email = ["my", "computer", "password", "is", "pizza"]
bob_input = np.array([[w2i[x] for x in bobs_email]])
bob_target = np.array([])

model = Embedding(vocab_size=len(vocab), dim=1)
model.weight.data *= 0

bobs_model = train(copy.deepcopy(model), bob_input, bob_target, iterations=1, batch_size=1)

# Peretas mengecek selisih bobot untuk menemukan kata-kata Bob
print("Kata-kata yang terdeteksi dari pembaruan bobot Bob:")
for i, v in enumerate(bobs_model.weight.data - model.weight.data):
    if v != 0:
        print(vocab[i])
3. Demonstrasi Homomorphic Encryption dengan Paillier (phe) (Buku hal. 289–290)
import phe

# 1. Membuat pasangan kunci publik & privat
public_key, private_key = phe.generate_paillier_keypair(n_length=1024)

# 2. Enkripsi dua nilai plaintext
x = public_key.encrypt(5)
y = public_key.encrypt(3)

# 3. Penjumlahan langsung di atas ciphertext (tanpa dekripsi)
z = x + y

# 4. Dekripsi hasil penjumlahan dengan kunci privat
z_decrypted = private_key.decrypt(z)
print("Hasil Penjumlahan Terenkripsi (Decrypted):", z_decrypted)

# **Secure Aggregation**

Solusi untuk mencegah pembocoran gradien individu adalah **Secure Aggregation**. Berdasarkan konsep *randomized response* dan *plausible deniability*, pembaruan gradien dari banyak pengguna dijumlahkan/diagregasikan terlebih dahulu sebelum ada pihak mana pun (termasuk pemilik server) yang dapat melihat nilai gradien tersebut.

# **Homomorphic Encryption**

**Homomorphic Encryption** adalah teknik kriptografi modern yang memungkinkan operasi matematika (seperti penjumlahan) dilakukan langsung di atas data yang terenkripsi (*ciphertext*) tanpa perlu mendekripsinya terlebih dahulu. Menggunakan pustaka Python `phe` (Paillier Encryption), kunci publik (*public key*) digunakan untuk mengenkripsi data, dan kunci privat (*private key*) digunakan untuk mendekripsi hasil penjumlahan akhir.

# **Homomorphically Encrypted Federated Learning**

Menggabungkan *Federated Learning* dengan *Homomorphic Encryption*. Pengguna (Bob, Alice, Sue) mengenkripsi bobot model lokal mereka menggunakan kunci publik sebelum saling membagikan gradien terenkripsi. Hasil penjumlahan gradien terenkripsi (*aggregated encrypted model*) dikirim kembali ke pemilik server. Pemilik server hanya dapat mendekripsi hasil penjumlahan akhir menggunakan kunci privat, sehingga tidak pernah bisa mengintip gradien individual milik masing-masing pengguna.

In [ ]:
import copy, phe
import numpy as np

model = Embedding(vocab_size=len(vocab), dim=1)
model.weight.data *= 0

# Penjanaan kunci Paillier
public_key, private_key = phe.generate_paillier_keypair(n_length=128)

def train_and_encrypt(model, input, target, pubkey):
    new_model = train(copy.deepcopy(model), input, target, iterations=1)
    encrypted_weights = list()
    for val in new_model.weight.data[:, 0]:
        encrypted_weights.append(pubkey.encrypt(val))
    ew = np.array(encrypted_weights).reshape(new_model.weight.data.shape)
    return ew

for i in range(3):
    print("\\nStarting Training Round...")
    print("\\tStep 1: send the model to Bob")
    bob_encrypted_model = train_and_encrypt(copy.deepcopy(model), bob, bob[22], public_key)

    print("\\n\\tStep 2: send the model to Alice")
    alice_encrypted_model = train_and_encrypt(copy.deepcopy(model), alice, alice[22], public_key)

    print("\\n\\tStep 3: Send the model to Sue")
    sue_encrypted_model = train_and_encrypt(copy.deepcopy(model), sue, sue[22], public_key)

    print("\\n\\tStep 4: Bob, Alice, and Sue send their encrypted models to each other.")
    aggregated_model = bob_encrypted_model + alice_encrypted_model + sue_encrypted_model

    print("\\n\\tStep 5: Only aggregated model is sent back to owner for decryption.")
    raw_values = list()
    for val in aggregated_model.flatten():
        raw_values.append(private_key.decrypt(val))

    new = np.array(raw_values).reshape(model.weight.data.shape) / 3
    model.weight.data = new
    print("\\t% Correct on Test Set: " + str(test(model, test_data, test_target) * 100))